In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
#read the csv from the training path and print first five rows
data_train = pd.read_csv("/content/drive/MyDrive/Data/processed_data_crossvalidation/train.csv")
print(data_train.head)

In [ ]:
LABEL_COLUMN = "label"

In [ ]:
# labels are there as string otherwise words
# need to convert the labels into numbers
print(data_train[LABEL_COLUMN].value_counts())
LABELS = list(data_train[LABEL_COLUMN].unique())
# sort the labels
LABELS.sort()
print(LABELS)

In [ ]:
# convert into numbers
data_train[LABEL_COLUMN] = pd.factorize(data_train[LABEL_COLUMN], sort = True)[0]
data_train[LABEL_COLUMN].value_counts()

In [ ]:
# get all features without the labels
# all the rows
# all the columns without the last column
X = data_train.drop(LABEL_COLUMN, axis = 1).values
# all rows
# only the lastb column, which is the label
Y = data_train[LABEL_COLUMN].values

In [ ]:
import threading, time

def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab import output
            output.eval_js("0")
        except: pass

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive started")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

In [ ]:
def cross_validation(model, data = (X, Y), splits = 3):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)
    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]
        # Fit the defined model
        model.fit(X_train, y_train)
        # Make predictions on the test data
        y_pred = model.predict(X_valid)
        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_pred, y_valid))
        precision.append(precision_score(y_pred, y_valid, average = 'macro'))
        recall.append(recall_score(y_pred, y_valid, average = 'macro'))
    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=1)
print("\nNeighbors: 1")
cross_validation(knn)

In [ ]:
#MLP
from sklearn.neural_network import MLPClassifier
lr = [0.000001,0.00001,0.0001,0.001,0.005,0.01, 0.05]
ep = [10,20,30,40,50]
for l in lr:
  for epoch in ep:
    print("\n Learning Rate:", l," epoch:", epoch)

    mlp = MLPClassifier(learning_rate_init=l, max_iter=epoch)
    cross_validation(mlp)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
max_depth = [6,7]
for depth in max_depth:
  print("\n trees:",80," depth:", depth)
  rf = RandomForestClassifier(n_estimators=80, max_depth= depth, n_jobs = -1)
  cross_validation(rf)

In [ ]:
from lightgbm import LGBMClassifier
depths = [1,2,3,4,5,6,7,8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
for depth in depths:
    for trees in n_estimators_list:
            print("\n" + "-"*50)
            print(f"Running LGBM with max_depth={depth}, n_estimators={trees}")
            print("-"*50)
            lgbm = LGBMClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001,verbosity=-1)
            cross_validation(lgbm)

In [ ]:
from xgboost import XGBClassifier
depths = [1,2,3,4,5,6,7,8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
for depth in depths:
    for trees in n_estimators_list:
            print("\n" + "-"*50)
            print(f"Running XGBoost with max_depth={depth}, n_estimators={trees}")
            print("-"*50)
            xgb = XGBClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001)
            cross_validation(xgb)

In [ ]:
#save the best model
from sklearn.ensemble import RandomForestClassifier
# Use the best hyperparameters found from the cross-validation (from cell sfHGvopINoGb)
# The output showed 'Mean Accuracy: 0.9967004425867967' for trees: 80, depth: 7
model = RandomForestClassifier(n_estimators=50, max_depth=12, n_jobs = -1)
# Fit the model on the entire training dataset
model.fit(X, Y)
# The lines for ypred, acctemp, and print(acctemp) are removed because X_train, Y_train, xValid, yValid are not defined
# in this scope. If evaluation on a separate test set is desired, that data needs to be loaded and split explicitly.
# For saving the best model, fitting on the entire training data (X, Y) with optimal hyperparameters is standard.

In [ ]:
import pickle
f = open("/content/drive/MyDrive/Model/Crossvalid_RF_50trees_12depth","wb")
pickle.dump(model,f)
f.close()

In [ ]:
#get features
features = data_train.drop('label', axis=1).columns

In [ ]:
#Top 10 important features
importances = model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': features,'Gini Importance': importances})
feature_imp_df =feature_imp_df.sort_values(by='Gini Importance', ascending=False)
print(feature_imp_df.head(10))

In [ ]:
#Save top10 in a df
top_10_features = feature_imp_df.head(10)
top_10_features = top_10_features.sort_values(by='Gini Importance', ascending=False)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(30, 10))
#plt.barh(features, importances, color='skyblue') #for all features
plt.barh(top_10_features['Feature'], top_10_features['Gini Importance'], color='blue') #for top 10 features
plt.xlabel('Gini Importance')
plt.title('Feature Importance - Gini Importance')
plt.gca().invert_yaxis()
plt.show()